# Project Milestone 5

In [546]:
# Importing Pandas packages required for this excercise.

import sqlite3
import pandas as pd
import seaborn as sns
import matplotlib

import urllib.request, urllib.parse, urllib.error
import requests
from bs4 import BeautifulSoup
import ssl
import re

import matplotlib.dates as mdates
from matplotlib.dates import DateFormatter
import matplotlib.ticker as tick

In [548]:
conn = sqlite3.connect("DSC540FinalProjectDB",timeout=10)

cursor = conn.cursor()

In [550]:
try:
    with sqlite3.connect("DSC540FinalProjectDB.db") as conn:
        cursor = conn.cursor()
        conn.commit()
        print("Succeeded")
except sqlite3.OperationalError as e:
    print("Failed to open database:", e)

Succeeded


In [552]:
# loading covid CSV file for data transformation and cleaning

covid = pd.read_csv("./time_series_covid_19_confirmed.csv")
covid = covid.rename(columns={'Country/Region': 'Country'})

covid.head()

,Province/State,Country,Lat,Long,1/22/20,1/23/20,1/24/20,1/25/20,1/26/20,1/27/20,...,4/4/20,4/5/20,4/6/20,4/7/20,4/8/20,4/9/20,4/10/20,4/11/20,4/12/20,4/13/20
0,NaN,Afghanistan,33.0000,65.0000,0,0,0,0,0,0,...,299,349,367,423,444,484,521,555,607,665
1,NaN,Albania,41.1533,20.1683,0,0,0,0,0,0,...,333,361,377,383,400,409,416,433,446,467
2,NaN,Algeria,28.0339,1.6596,0,0,0,0,0,0,...,1251,1320,1423,1468,1572,1666,1761,1825,1914,1983
3,NaN,Andorra,42.5063,1.5218,0,0,0,0,0,0,...,466,501,525,545,564,583,601,601,638,646
4,NaN,Angola,-11.2027,17.8739,0,0,0,0,0,0,...,10,14,16,17,19,19,19,19,19,19


In [554]:
# Selecting 4 countries that has highest number of cases

covidMelt = covid.melt(id_vars = ['Country'])
coviddropNan = covidMelt.dropna()
coviddropvariable = coviddropNan[coviddropNan['variable'] != 'Province/State']
covidCountries = coviddropvariable.groupby(['Country'])['value'].sum().reset_index().sort_values(by = 'value',ascending = False)
covidCountries = covidCountries.rename(columns={'value': 'Covidcases'})
covidCountries[0:5] 

,Country,Covidcases
171,US,6278184.3773
36,China,5267388.7564
84,Italy,2977275.0
156,Spain,2574218.0
65,Germany,1986377.0


In [556]:
# Write the covid countries DataFrame to a SQLite table
covidCountries.to_sql("covidCountries", conn, if_exists="replace")

185

In [558]:
# reading covidtopCountries table data in sqlite database
res = conn.execute("SELECT * FROM covidCountries")
for row in res:
    print(row)

(171, 'US', 6278184.3773)
(36, 'China', 5267388.7564)
(84, 'Italy', 2977275.0)
(156, 'Spain', 2574218.0)
(65, 'Germany', 1986377.0)
(61, 'France', 1748435.0816000002)
(80, 'Iran', 1329967.0)
(175, 'United Kingdom', 923997.1052)
(170, 'Turkey', 526623.2069999999)
(161, 'Switzerland', 445000.0457)
(90, 'Korea, South', 390965.0)
(16, 'Belgium', 379608.8333)
(120, 'Netherlands', 362341.0139)
(32, 'Canada', 283292.1318)
(9, 'Austria', 248548.0663)
(23, 'Brazil', 228935.8397)
(135, 'Portugal', 212170.1754)
(83, 'Israel', 150816.0)
(160, 'Sweden', 147645.0)
(126, 'Norway', 128255.9409)
(138, 'Russia', 125270.0)
(8, 'Australia', 112560.8928)
(82, 'Ireland', 105561.4503)
(46, 'Denmark', 99170.84909999999)
(45, 'Czechia', 90898.2905)
(86, 'Japan', 86311.0)
(35, 'Chile', 85974.7819)
(78, 'India', 82647.0)
(105, 'Malaysia', 79299.0)
(51, 'Ecuador', 79100.9854)
(134, 'Poland', 78867.06450000001)
(137, 'Romania', 73497.91)
(128, 'Pakistan', 65871.72039999999)
(132, 'Peru', 61137.7948)
(133, 'Philipp

In [560]:
# loading covid CSV file for data transformation and cleaning

url = 'https://en.wikipedia.org/wiki/COVID-19_pandemic_by_country_and_territory'

In [562]:
dfs = pd.read_html(url)

In [563]:
corona_df=dfs[12]
corona_df_col=corona_df.drop(columns=['Unnamed: 0'])

In [564]:
# remove records where deaths are equals to 0 or less
corona_df_clean=corona_df_col[(corona_df_col['Deaths'])>"0"]

In [565]:
# Write the covid DataFrame to a SQLite table
corona_df_clean.to_sql("coronawebdata", conn, if_exists="replace")

232

In [570]:
def read_json_from_web_api(url):
    """
    Reads a JSON file from a web API.

    Args:
        url (str): The URL of the web API endpoint.

    Returns:
        dict: The JSON data as a Python dictionary, or None if an error occurs.
    """
    try:
        response = requests.get(url)
        response.raise_for_status()  # Raise HTTPError for bad responses (4xx or 5xx)
        return response.json()
    except requests.exceptions.RequestException as e:
        print(f"Request error: {e}")
        return None
    except json.JSONDecodeError as e:
        print(f"JSON decode error: {e}")
        return None

In [572]:
# call Covid 19 API and load it in Json Format
url = "https://api.covidtracking.com/v1/us/daily.json"  # Replace with the actual API endpoint
data = read_json_from_web_api(url)
df = pd.json_normalize(data)
df = df.drop('hash',axis=1)    
df.head(10)

,date,states,positive,negative,pending,hospitalizedCurrently,hospitalizedCumulative,inIcuCurrently,inIcuCumulative,onVentilatorCurrently,...,totalTestResults,lastModified,recovered,total,posNeg,deathIncrease,hospitalizedIncrease,negativeIncrease,positiveIncrease,totalTestResultsIncrease
0,20210307,56,28756489.0,74582825.0,11808.0,40199.0,776361.0,8134.0,45475.0,2802.0,...,363825123,2021-03-07T24:00:00Z,None,0,0,842,726,131835,41835,1170059
1,20210306,56,28714654.0,74450990.0,11783.0,41401.0,775635.0,8409.0,45453.0,2811.0,...,362655064,2021-03-06T24:00:00Z,None,0,0,1680,503,143835,60015,1430992
2,20210305,56,28654639.0,74307155.0,12213.0,42541.0,775132.0,8634.0,45373.0,2889.0,...,361224072,2021-03-05T24:00:00Z,None,0,0,2221,2781,271917,68787,1744417
3,20210304,56,28585852.0,74035238.0,12405.0,44172.0,772351.0,8970.0,45293.0,2973.0,...,359479655,2021-03-04T24:00:00Z,None,0,0,1743,1530,177957,65487,1590984
4,20210303,56,28520365.0,73857281.0,11778.0,45462.0,770821.0,9359.0,45214.0,3094.0,...,357888671,2021-03-03T24:00:00Z,None,0,0,2449,2172,267001,66836,1406795
5,20210302,56,28453529.0,73590280.0,11196.0,46388.0,768649.0,9465.0,45084.0,3169.0,...,356481876,2021-03-02T24:00:00Z,None,0,0,1728,1871,255779,54248,1343519
6,20210301,56,28399281.0,73334501.0,11748.0,46738.0,766778.0,9595.0,44956.0,3171.0,...,355138357,2021-03-01T24:00:00Z,None,0,0,1241,1024,118077,48092,1154440
7,20210228,56,28351189.0,73216424.0,11708.0,47352.0,765754.0,9802.0,44907.0,3245.0,...,353983917,2021-02-28T24:00:00Z,None,0,0,1051,879,203599,54349,1408422
8,20210227,56,28296840.0,73012825.0,11731.0,48871.0,764875.0,10114.0,44875.0,3335.0,...,352575495,2021-02-27T24:00:00Z,None,0,0,1847,1428,205090,71245,1655179
9,20210226,56,28225595.0,72807735.0,11945.0,51112.0,763447.0,10466.0,44791.0,3466.0,...,350920316,2021-02-26T24:00:00Z,None,0,0,2141,1868,276829,74857,1803309


In [573]:
df_clean = df.dropna(subset=['positive', 'negative','pending','hospitalizedCurrently','hospitalizedCumulative','inIcuCurrently','inIcuCumulative','onVentilatorCurrently'])
print(df_clean)

         date  states    positive    negative  pending  hospitalizedCurrently  \
0    20210307      56  28756489.0  74582825.0  11808.0                40199.0   
1    20210306      56  28714654.0  74450990.0  11783.0                41401.0   
2    20210305      56  28654639.0  74307155.0  12213.0                42541.0   
3    20210304      56  28585852.0  74035238.0  12405.0                44172.0   
4    20210303      56  28520365.0  73857281.0  11778.0                45462.0   
..        ...     ...         ...         ...      ...                    ...   
342  20200330      56    171965.0    416268.0  65369.0                15892.0   
343  20200329      56    150826.0    366103.0  65404.0                14055.0   
344  20200328      56    131143.0    324271.0  65698.0                12393.0   
345  20200327      56    111358.0    255808.0  59795.0                10887.0   
346  20200326      56     92143.0    210652.0  60131.0                 7805.0   

     hospitalizedCumulative

In [576]:
# Write the covid web DataFrame to a SQLite table
corona_df_clean.to_sql("cleaneddfweb", conn, if_exists="replace")

232

In [578]:
# pushing Covid API Dataframe into Sql Table
df_clean.to_sql("cleanedAPI", conn, if_exists="replace")

347

## Join covidcountries and Cleaneddfweb tables to get death values by countries

In [580]:
sql = '''
            SELECT cc.Country,cc.Covidcases,cf.Deaths
            FROM covidCountries AS cc 
                INNER JOIN cleaneddfweb AS cf ON cf.Country = cc.Country
            ORDER BY cc.Country
            
        '''

cursor.execute(sql)

# Fetching rows from the result table
result = cursor.fetchall()
for row in result:
    print(row)

('Afghanistan', 6828.0, '7998')
('Albania', 7158.3216, '3605')
('Algeria', 24044.6935, '6881')
('Andorra', 9758.0281, '159')
('Angola', 249.6712, '1937')
('Antigua and Barbuda', 233.2644, '146')
('Argentina', 27772.9672, '130663')
('Armenia', 15190.1073, '8777')
('Australia', 112560.8928, '25236')
('Austria', 248548.0663, '22534')
('Azerbaijan', 11723.720000000001, '10353')
('Bahamas', 495.63800000000003, '849')
('Bahrain', 19019.5775, '1536')
('Bangladesh', 4154.0413, '29499')
('Barbados', 977.6507, '593')
('Belarus', 17185.6632, '7118')
('Belgium', 379608.8333, '34339')
('Belize', 76.6507, '688')
('Benin', 403.62350000000004, '163')
('Bhutan', 227.9478, '21')
('Bolivia', 3292.1211000000003, '22387')
('Bosnia and Herzegovina', 12729.595000000001, '16392')
('Botswana', 114.35640000000001, '2801')
('Brazil', 228935.8397, '702116')
('Brunei', 3559.263, '179')
('Bulgaria', 11035.2197, '38700')
('Burkina Faso', 6972.6767, '400')
('Burundi', 72.5458, '15')
('Cabo Verde', 135.497, '417')
('C

## Get Year = 2021 data for covid i.e. positive, negative and pending reports for covid patient.

In [585]:
sql = '''
            SELECT substr(date,1,4) as APIDate,positive,negative,pending,hospitalizedCurrently
            FROM cleanedAPI AS cf 
            WHERE substr(date,1,4) = '2021'
                        
        '''

cursor.execute(sql)

# Fetching rows from the result table
result = cursor.fetchall()
for row in result:
    print(row)

('2021', 28756489.0, 74582825.0, 11808.0, 40199.0)
('2021', 28714654.0, 74450990.0, 11783.0, 41401.0)
('2021', 28654639.0, 74307155.0, 12213.0, 42541.0)
('2021', 28585852.0, 74035238.0, 12405.0, 44172.0)
('2021', 28520365.0, 73857281.0, 11778.0, 45462.0)
('2021', 28453529.0, 73590280.0, 11196.0, 46388.0)
('2021', 28399281.0, 73334501.0, 11748.0, 46738.0)
('2021', 28351189.0, 73216424.0, 11708.0, 47352.0)
('2021', 28296840.0, 73012825.0, 11731.0, 48871.0)
('2021', 28225595.0, 72807735.0, 11945.0, 51112.0)
('2021', 28150738.0, 72530906.0, 11813.0, 52669.0)
('2021', 28075173.0, 72258697.0, 12548.0, 54118.0)
('2021', 28001915.0, 72013379.0, 11200.0, 55058.0)
('2021', 27932810.0, 71788112.0, 9499.0, 55403.0)
('2021', 27880280.0, 71664501.0, 9442.0, 56159.0)
('2021', 27821578.0, 71507723.0, 9408.0, 58222.0)
('2021', 27749224.0, 71365933.0, 8711.0, 59882.0)
('2021', 27674548.0, 71141178.0, 8679.0, 62224.0)
('2021', 27607724.0, 70922687.0, 8548.0, 63329.0)
('2021', 27540885.0, 70689021.0, 1154

## Grouping covid metrics by year.

In [589]:
sql = '''
            SELECT substr(date,1,4) as APIDate,SUM(positive) as positivecases,
            SUM(negative) as negativecases,AVG(pending) as pendingcases,AVG(hospitalizedCurrently) as hospitalized
            FROM cleanedAPI AS cf 
            GROUP BY substr(date,1,4) 
                        
        '''

cursor.execute(sql)

# Fetching rows from the result table
result = cursor.fetchall()
for row in result:
    print(row)

('2020', 1711472650.0, 6828680724.0, 9640.38078291815, 52113.55516014235)
('2021', 1689066577.0, 4443463114.0, 11093.151515151516, 90629.54545454546)
